In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# Import necesssary modules
import numpy as np
from keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding
import tensorflow as tf

In [1]:
'''The following code snippet loads data, tokenizes and pads sequences of source code and summaries to feed a model.

Parameters:

/content/drive/MyDrive/source_code_summarization_full_data/data_ps.declbodies: A string representing the path to the file containing the source code data.
/content/drive/MyDrive/source_code_summarization_full_data/data_ps.descriptions: A string representing the path to the file containing the summaries data.

max_code_length: An integer representing the maximum length of the code sequences after padding.
max_summary_length: An integer representing the maximum length of the summary sequences after padding.
Returns:

padded_code: A numpy array of shape (number of sequences, max_code_length) representing the padded and tokenized code sequences.
padded_summaries: A numpy array of shape (number of sequences, max_summary_length) representing the padded and tokenized summary sequences.
vocab_code_size: An integer representing the vocabulary size of the source code tokenizer plus one for the padding token.
vocab_summary_size: An integer representing the vocabulary size of the summary tokenizer plus one for the padding token.
Note: This code assumes that the input data is formatted as follows: each line of the source code file corresponds
 to a single source code sequence and each line of the summaries file corresponds to a single summary sequence.'''


# Read the declbodies file
with open("data_ps.declbodies", "r", encoding='utf-8') as declbodies_file:
    source_code_data = [line.strip() for line in declbodies_file]

# Read the descriptions file
start_token = "<start>"
with open("data_ps.descriptions", "r", encoding='utf-8') as descriptions_file:
    summaries_data = [f"{start_token} {line.strip()}" for line in descriptions_file]

# Tokenize the source code and summaries
tokenizer_code = Tokenizer(filters='', lower=False)
tokenizer_code.fit_on_texts(source_code_data)
sequences_code = tokenizer_code.texts_to_sequences(source_code_data)

# Tokenize the summaries
tokenizer_summaries = Tokenizer(filters='', lower=False)
tokenizer_summaries.fit_on_texts(summaries_data)
sequences_summaries = tokenizer_summaries.texts_to_sequences(summaries_data)

# Pad the sequences
max_summary_length = max([len(seq) for seq in sequences_summaries])
padded_summaries = pad_sequences(sequences_summaries, maxlen=max_summary_length, padding='post')

max_code_length = 500  # Choose an appropriate value based on your data
max_summary_length = 200  # Choose an appropriate value based on your data

padded_code = pad_sequences(sequences_code, maxlen=max_code_length, padding='post', truncating='post')
padded_summaries = pad_sequences(sequences_summaries, maxlen=max_summary_length, padding='post', truncating='post')

vocab_code_size = len(tokenizer_code.word_index) + 1
vocab_summary_size = len(tokenizer_summaries.word_index) + 1


NameError: name 'Tokenizer' is not defined

In [8]:
'''This code defines a Keras model for a sequence-to-sequence neural machine translation task
 using an encoder-decoder architecture with dropout regularization. The model consists of 
 an encoder network and a decoder network. The encoder network takes as input a sequence 
 of integers representing the source code, and outputs a representation of the input sequence. 
 The encoder is a stack of LSTM layers, and the output of the last LSTM layer is passed through
a dropout layer with a dropout rate of 0.2 to obtain the encoder states.

The decoder network takes as input a sequence of integers representing the summary, 
and the initial states of the decoder are set to the final states of the encoder. 
The decoder is also a stack of LSTM layers with a dropout rate of 0.2 applied to
 the input of the LSTM layers. The output of the last LSTM layer is passed through 
 a dense layer with a softmax activation function to obtain the predicted summary.

The final model is created by specifying the inputs to the model (encoder input and decoder input)
 and the output (decoder output). The model is returned as an instance of the Keras Model class.

Args:
embedding_dim (int): dimensionality of the embedding layer.
lstm_units (int): dimensionality of the LSTM layers.

Returns:
A Keras Model instance representing the encoder-decoder model.'''

embedding_dim = 128
lstm_units = 50

from keras.layers import Dropout

# Encoder
encoder_inputs = Input(shape=(max_code_length,))
encoder_embedding = Embedding(vocab_code_size, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(lstm_units, return_state=True)
encoder_dropout = Dropout(0.2) # Add a dropout layer with rate 0.2
_, encoder_state_h, encoder_state_c = encoder_lstm(encoder_embedding)
encoder_dropout_output = encoder_dropout(encoder_state_h) # Apply dropout to the output
encoder_states = [encoder_dropout_output, encoder_state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_summary_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_dropout = Dropout(0.2) # Add a dropout layer with rate 0.2
decoder_dropout_output = decoder_dropout(decoder_embedding) # Apply dropout to the input
decoder_outputs, _, _ = decoder_lstm(decoder_dropout_output, initial_state=encoder_states)
decoder_dense = Dense(vocab_summary_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)


In [9]:
'''This code compiles and trains a Keras model using the sparse categorical cross-entropy loss function. 
The training data is passed as two inputs: the tokenized source code and the tokenized summaries 
with the last token removed. The target summary sequences are created by shifting the input summary
 sequences one time step forward. The model is trained for a specified number of epochs with a 
 specified batch size and a validation split of 20%.

Input:

padded_code : Numpy array of tokenized source code sequences, padded to a fixed length.
padded_summaries : Numpy array of tokenized summary sequences with the last token removed, padded to a fixed length.
Output:

Trained Keras model with the specified number of epochs and batch size.'''

batch_size = 32
epochs = 30

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit([padded_code, padded_summaries[:, :-1]], np.expand_dims(padded_summaries[:, 1:], -1),
          batch_size=batch_size, epochs=epochs, validation_split=0.2)

Epoch 1/30
1487/1487 [==============================] - 348s 228ms/step - loss: 1.9516 - val_loss: 1.3352
Epoch 2/30
1487/1487 [==============================] - 231s 156ms/step - loss: 1.2601 - val_loss: 1.2459
Epoch 3/30
1487/1487 [==============================] - 216s 145ms/step - loss: 1.1690 - val_loss: 1.1980
Epoch 4/30
1487/1487 [==============================] - 214s 144ms/step - loss: 1.1064 - val_loss: 1.1688
Epoch 5/30
1487/1487 [==============================] - 206s 138ms/step - loss: 1.0598 - val_loss: 1.1507
Epoch 6/30
1487/1487 [==============================] - 208s 140ms/step - loss: 1.0219 - val_loss: 1.1387
Epoch 7/30
1487/1487 [==============================] - 206s 139ms/step - loss: 0.9884 - val_loss: 1.1280
Epoch 8/30
1487/1487 [==============================] - 203s 137ms/step - loss: 0.9573 - val_loss: 1.1192
Epoch 9/30
1487/1487 [==============================] - 201s 135ms/step - loss: 0.9293 - val_loss: 1.1133
Epoch 10/30
1487/1487 [=======================

In [12]:
'''This code snippet defines an encoder-decoder model for generating summaries from source code. 
The encoder and decoder models are defined using the Keras functional API. 
The decoder model is defined to take the encoded input sequence and generate a summary by 
predicting the next token in the sequence using the previously generated tokens. 
The reverse lookup dictionaries are used to convert the predicted tokens back into text. 
The decode_sequence function takes the preprocessed input sequence, generates the summary 
using the decoder model, and returns the generated summary as a string.
Finally, the input source code is tokenized and padded using the tokenizer_code and max_code_length variables, 
and the decode_sequence function is called with the preprocessed input to generate the summary.'''

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(lstm_units,))
decoder_state_input_c = Input(shape=(lstm_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inference = Embedding(vocab_summary_size, embedding_dim)(decoder_inputs)
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding_inference, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)


# Reverse-lookup token index to decode sequences back to something readable.
reverse_input_char_index = dict((i, char) for char, i in tokenizer_code.word_index.items())
reverse_target_char_index = dict((i, char) for char, i in tokenizer_summaries.word_index.items())

def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0] = tokenizer_summaries.word_index["<start>"]

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        
        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        
        # Check if the sampled_token_index is in the reverse_target_char_index dictionary
        if sampled_token_index in reverse_target_char_index:
            sampled_char = reverse_target_char_index[sampled_token_index]
            decoded_sentence += " " + sampled_char
        else:
            print(f"Sampled token index {sampled_token_index} not found in reverse_target_char_index dictionary.")
        
        # Exit condition: either hit max length or find the stop token.
        if sampled_char == "<end>" or len(decoded_sentence) > max_summary_length:
            stop_condition = True
        
        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        
        # Update states
        states_value = [h, c]
    
    return decoded_sentence

# Input source code as a string
input_source_code = '''def _first_batch(sock_info, db, coll, query, ntoreturn, slave_ok, codec_options, read_preference, cmd, listeners): DCNL  DCSP query = _Query(0, db, coll, 0, query, None, codec_options, read_preference, ntoreturn, 0, DEFAULT_READ_CONCERN, None) DCNL DCSP name = next(iter(cmd)) DCNL DCSP duration = None DCNL DCSP publish = listeners.enabled_for_commands DCNL DCSP if publish: DCNL DCSP  DCSP start = datetime.datetime.now() DCNL DCSP (request_id, msg, max_doc_size) = query.get_message(slave_ok, sock_info.is_mongos) DCNL DCSP if publish: DCNL DCSP  DCSP encoding_duration = (datetime.datetime.now() - start) DCNL DCSP  DCSP listeners.publish_command_start(cmd, db, request_id, sock_info.address) DCNL DCSP  DCSP start = datetime.datetime.now() DCNL DCSP sock_info.send_message(msg, max_doc_size) DCNL DCSP response = sock_info.receive_message(1, request_id) DCNL DCSP try: DCNL DCSP  DCSP result = _unpack_response(response, None, codec_options) DCNL DCSP except Exception as exc: DCNL DCSP  DCSP if publish: DCNL DCSP  DCSP  DCSP duration = ((datetime.datetime.now() - start) + encoding_duration) DCNL DCSP  DCSP  DCSP if isinstance(exc, (NotMasterError, OperationFailure)): DCNL DCSP  DCSP  DCSP  DCSP failure = exc.details DCNL DCSP  DCSP  DCSP else: DCNL DCSP  DCSP  DCSP  DCSP failure = _convert_exception(exc) DCNL DCSP  DCSP  DCSP listeners.publish_command_failure(duration, failure, name, request_id, sock_info.address) DCNL DCSP  DCSP raise DCNL DCSP if publish: DCNL DCSP  DCSP duration = ((datetime.datetime.now() - start) + encoding_duration) DCNL DCSP  DCSP listeners.publish_command_success(duration, result, name, request_id, sock_info.address) DCNL DCSP return result
def disk_partitions(all=False): DCNL  DCSP result = [dict(partition._asdict()) for partition in psutil.disk_partitions(all)] DCNL DCSP return result
def test_find_module_4(): DCNL  DCSP nt.assert_is_none(mp.find_module(None))'''  # Replace with your desired source code string

# Tokenize the input source code string
input_sequence = tokenizer_code.texts_to_sequences([input_source_code])

# Pad the tokenized input sequence
padded_input_sequence = pad_sequences(input_sequence, maxlen=max_code_length, padding='post')

# Call the decode_sequence function with the preprocessed input sequence
input_seq = padded_input_sequence[0:1]  # Extract the first sequence from the preprocessed input
generated_summary = decode_sequence(input_seq)
print("Generated summary:", generated_summary)

1/1 [==============================] - 0s 24ms/step
Generated summary:  function of for for DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL DCNL


In [13]:
'''This code snippet saves a trained model and associated tokenizers to disk. 
The trained model is saved as an H5 file with the name "trained_model.h5". 
The tokenizers, which are Python objects that have been trained on the code and summary data, are saved using the pickle module. 
The tokenizer for the code data is saved to a file called "tokenizer_code.pickle", 
and the tokenizer for the summary data is saved to a file called "tokenizer_summaries.pickle". 
The protocol parameter is set to pickle.HIGHEST_PROTOCOL, which ensures that the highest protocol 
version available is used for pickling the objects.'''


import pickle

# Save the trained model
model.save("trained_model.h5")

# Save the tokenizers
with open("tokenizer_code.pickle", "wb") as handle:
    pickle.dump(tokenizer_code, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open("tokenizer_summaries.pickle", "wb") as handle:
    pickle.dump(tokenizer_summaries, handle, protocol=pickle.HIGHEST_PROTOCOL)
